***

Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft

pd.options.display.float_format = '{:.1f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_out  = os.path.join(path_users
                             , 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents'
                             , 'Process Revamp'
                             , 'Task 9. Collect new data'
                             , 'Census')

path_code    = os.path.join(path_git, 'Data', 'EIA')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://www.eia.gov/opendata/documentation.php
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

***

Preparing API Request

***

In [ ]:
# Execute script to prepare API request inputs
exec(open(os.path.join(path_code, 'config', 'Configuration File.py')).read())

***

Importing

***

In [ ]:
cat         = df_api['category'   ].values[0]
route1      = df_api['route1'     ].values[0]
route2      = df_api['route2'     ].values[0]
facetOption = df_api['facetOption'].values[0]
facet       = df_api['facet'      ].values[0]
freq = 'annual'

URL

In [ ]:
print('')
print('')
print('API request: ')
print('')
print(cat)
print(route1)
print(route2)
print(facetOption)
print(facet)
print('')


root_ = 'https://api.eia.gov/v2'

route_ = f'/{cat}/{route1}/{route2}'
data_ = f'/data/?frequency={freq}&data[0]=value&facets[{facetOption}][]={facet}'
api_key_ = f'&api_key={api_key}'

    
url = f"{root_}{route_}{data_}{api_key_}"

# Use requests package to call out to the API
response = requests.get(url).text
response = response.replace('null', '"null"')
response = ast.literal_eval(response)

print('')
print('API response: ')
print('')

response

In [ ]:
list_df = []

for row in range(len(response['response']['data'])):
    list_df.append(pd.DataFrame(response['response']['data'][row], index = [0]))

df_eia = pd.concat(list_df)
df_eia = df_eia.sort_values('period')
df_eia = df_eia.reset_index(drop = True)

df_eia.head()

Headers

In [ ]:
print('')
print('')
print('Your request: ')
print('')
print(cat)
print(route1)
print(route2)
print(facetOption)
print(facet)
print('')


params = {
    'api_key': api_key,
    "frequency": freq,
    "data[0]": 'value',
    f'facets[{facetOption}][]': facet,
    'start': 2000
}

root_ = 'https://api.eia.gov/v2'
route_ = f'/{cat}/{route1}/{route2}'
data_ = f'/data'

url = f"{root_}{route_}{data_}"


# Use requests package to call out to the API
response = requests.get(url, params=params)
response = response.json()

print('')
print('API response: ')
print('')

response

In [ ]:
list_df = []

for row in range(len(response['response']['data'])):
    list_df.append(pd.DataFrame(response['response']['data'][row], index = [0]))

df_eia = pd.concat(list_df)
df_eia = df_eia.sort_values('period')
df_eia = df_eia.reset_index(drop = True)

df_eia.head()

In [ ]:
# df_road1 = df_road1[df_road1['product-name'] == 'Regular Gasoline']
